Imports, paths, helpers

In [1]:
# ==== config & helpers ====
import os, re, json, math
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

FINAL = Path("data/final")
FINAL.mkdir(parents=True, exist_ok=True)

def log(x): print(x)

def safe_lower(s):
    try: return str(s).lower()
    except: return ""

def choose_sentiment(row):
    """
    Prefer an existing sentiment label/score if present (from Shubham's pipeline),
    else derive from rating (>=4 positive, <=2 negative, else neutral).
    Returns (label, confidence in [0,1]).
    """
    # Try Shubham-like columns first
    if "sentiment_label" in row and pd.notna(row["sentiment_label"]):
        label = str(row["sentiment_label"]).strip().lower()
        if "avg_sentiment" in row and pd.notna(row["avg_sentiment"]):
            try:
                conf = float(row["avg_sentiment"])
                conf = min(1.0, max(0.0, abs(conf)))
            except:
                conf = 0.6
        else:
            conf = 0.6
        return label, conf

    # Fallback: rating -> label
    r = None
    if "rating" in row and pd.notna(row["rating"]):
        try: r = float(row["rating"])
        except: r = None
    if r is not None:
        if r >= 4.0:   return "positive", 0.7
        if r <= 2.0:   return "negative", 0.7
        return "neutral", 0.5

    return "neutral", 0.4


Aspect dictionaries (tweak anytime)

In [3]:
# ==== aspect keyword dictionaries by place category ====

ASPECTS = {
    "restaurant": {
        "food":       ["food","meal","dish","burger","pizza","pasta","menu","taste","flavour","flavor","portion","fresh","stale","spicy","sweet","salty"],
        "service":    ["service","staff","waiter","waitress","server","attentive","rude","slow","friendly","helpful"],
        "ambience":   ["ambience","ambiance","atmosphere","music","noisy","quiet","vibe","decor","seating","crowded","cozy","cozy"],
        "price":      ["price","priced","expensive","cheap","value","cost","affordable","overpriced","deal"],
        "cleanliness":["clean","dirty","hygiene","messy","tidy","sticky","smell","smelly","stink"],
    },
    "park": {
        "facilities": ["toilet","bbq","playground","bench","picnic","parking","carpark","car park"],
        "cleanliness":["clean","litter","rubbish","trash","dirty","tidy"],
        "safety":     ["safe","danger","dangerous","crime","lighting","dark"],
        "crowd":      ["crowd","busy","packed","quiet","peaceful","calm"],
        "scenery":    ["view","scenery","lake","beach","trees","nature","wildlife","flowers","sunset"],
        "access":     ["access","wheelchair","path","trail","walkway","cycle","bike"],
    },
    "shopping mall": {
        "stores":     ["store","shop","brands","variety","selection"],
        "parking":    ["parking","carpark","car park","garage","spaces"],
        "crowd":      ["crowded","busy","queue","line","packed"],
        "cleanliness":["clean","dirty","tidy","smell","smelly"],
        "food court": ["food court","foodcourt","eatery","restaurant","cafe"],
    },
    "tourist attraction": {
        "experience": ["experience","tour","guide","exhibit","museum","ride","activity","photo","view"],
        "crowd":      ["queue","line","wait","crowded","busy","packed"],
        "price":      ["ticket","price","expensive","cheap","value"],
        "facilities": ["toilet","parking","shop","cafe","access","wheelchair"],
        "staff":      ["staff","guide","host","friendly","rude","helpful"],
    },
}

# default fallback if category unknown
DEFAULT_ASPECTS = {
    "experience": ["experience","visit","trip","time","moment","memorable","boring"],
    "service":    ["service","staff","team","support","helpful","rude","friendly"],
    "price":      ["price","expensive","cheap","value","cost"],
    "cleanliness":["clean","dirty","tidy","messy","smell","smelly"],
}


Load reviews & (optionally) places to pick a category

In [5]:
# ==== load inputs ====
rev_path = FINAL/"reviews.csv"
pl_path  = FINAL/"places.csv"  # used to get category per place if available

assert rev_path.exists(), f"Missing {rev_path}"
reviews = pd.read_csv(rev_path)

places = pd.DataFrame()
if pl_path.exists():
    places = pd.read_csv(pl_path)
    # normalise column names a bit
    if "category" not in places.columns and "types" in places.columns:
        places["category"] = places["types"].astype(str).str.extract(r"(restaurant|park|shopping mall|tourist attraction)", expand=False)
    places = places[["place_id","category"]].dropna().drop_duplicates()

# join category if we have it
if not places.empty and "place_id" in reviews.columns:
    reviews = reviews.merge(places, on="place_id", how="left")

# ensure essential columns
for c in ["review_id","place_id","text","rating","category"]:
    if c not in reviews.columns:
        reviews[c] = None

# clean text
reviews["text_lc"] = reviews["text"].apply(safe_lower)
log(f"reviews loaded: {reviews.shape}")


reviews loaded: (28917, 15)


Extract aspects + assign sentiment

In [10]:
# ==== aspect extraction ====
rows = []

for _, r in reviews.iterrows():
    text = r["text_lc"] or ""
    if not text:
        continue

    # safe category handling
    raw_cat = r.get("category")
    if pd.isna(raw_cat):
        cat = ""
    else:
        cat = str(raw_cat).strip().lower()

    aspect_dict = ASPECTS.get(cat, DEFAULT_ASPECTS)

    found_aspects = set()
    for aspect, kws in aspect_dict.items():
        for kw in kws:
            # word boundary for single words; allow spaces for multi-words
            pattern = r"\b" + re.escape(kw) + r"\b" if " " not in kw else re.escape(kw)
            if re.search(pattern, text):
                found_aspects.add(aspect)
                break


    # sentiment for this review (used for each aspect hit)
    label, conf = choose_sentiment(r)

    # if no aspect keyword found, you can optionally attach a generic one
    if not found_aspects:
        # comment the next line if you prefer to skip “no-aspect” reviews
        found_aspects = {"experience"}

    for a in sorted(found_aspects):
        rows.append({
            "review_id": r.get("review_id"),
            "place_id":  r.get("place_id"),
            "aspect":    a,
            "sentiment": label,
            "confidence": round(float(conf or 0), 3),
        })

aspects_df = pd.DataFrame(rows)
log(f"aspects extracted: {aspects_df.shape}")


aspects extracted: (36278, 5)


Save aspects.csv + tiny quality report

In [12]:
# ==== save output ====
out_path = FINAL/"aspects.csv"
if aspects_df.empty:
    # write scaffold to avoid missing-file surprises in CI/frontend
    aspects_df = pd.DataFrame(columns=["review_id","place_id","aspect","sentiment","confidence"])

aspects_df.to_csv(out_path, index=False)
log(f"wrote {out_path}, rows={len(aspects_df)}")

# quick QA snapshot (prints in CI logs)
if not aspects_df.empty:
    print("\nTop aspects:")
    print(aspects_df["aspect"].value_counts().head(10))
    print("\nAspect x sentiment (sample):")
    ctab = pd.crosstab(aspects_df["aspect"], aspects_df["sentiment"])
    print(ctab.head(10))


wrote data\final\aspects.csv, rows=36278

Top aspects:
aspect
experience     18977
service        10460
price           2628
cleanliness     1870
facilities       703
scenery          443
crowd            315
access           205
food             189
staff            145
Name: count, dtype: int64

Aspect x sentiment (sample):
sentiment    negative  neutral  positive
aspect                                  
access              8       16       181
ambience            2        9        79
cleanliness       231      144      1495
crowd               9       25       281
experience       3240     1240     14497
facilities         25       57       621
food               10       13       166
food court          5        3        22
parking             8        5        55
price             510      221      1897
